In [ ]:
!pip install numpy numba matplotlib

In [ ]:
# Q1
import numpy as np
from numba import cuda
import time
import math

N = 5_000_000

def f_cpu(arr):
    return arr**2 + 3*arr + 5

@cuda.jit
def f_cuda(arr, out):
    i = cuda.grid(1)
    if i < arr.size:
        out[i] = arr[i]**2 + 3*arr[i] + 5

def benchmark_type(dtype):
    print(f'\n--- Benchmarking {dtype} ---')
    arr = np.random.rand(N).astype(dtype)
    out = np.zeros_like(arr)
    start = time.time()
    out_cpu = f_cpu(arr)
    cpu_time = time.time() - start
    print(f'CPU Time: {cpu_time:.5f} s')
    d_arr = cuda.to_device(arr)
    d_out = cuda.device_array_like(arr)
    threadsperblock = 256
    blockspergrid = math.ceil(arr.size / threadsperblock)
    f_cuda[blockspergrid, threadsperblock](d_arr, d_out)
    cuda.synchronize()
    start = time.time()
    f_cuda[blockspergrid, threadsperblock](d_arr, d_out)
    cuda.synchronize()
    gpu_time = time.time() - start
    print(f'GPU Time: {gpu_time:.5f} s')
    print(f'Speedup: {cpu_time/gpu_time:.2f}x')

benchmark_type(np.float32)
benchmark_type(np.float64)

In [ ]:
# Q2
import numpy as np
from numba import njit
import time

N = 1_000_000
data = np.random.rand(N)
bins = 100

def hist_python(data, bins):
    counts = [0] * bins
    for val in data:
        idx = int(val * bins)
        if idx == bins: idx -= 1
        counts[idx] += 1
    return counts

@njit
def hist_numba(data, bins):
    counts = np.zeros(bins, dtype=np.int64)
    for val in data:
        idx = int(val * bins)
        if idx == bins: idx -= 1
        counts[idx] += 1
    return counts

start = time.time()
_ = np.histogram(data, bins=bins, range=(0.0, 1.0))[0]
print(f'NumPy Time: {time.time() - start:.5f} s')

start = time.time()
_ = hist_python(data.tolist(), bins)
print(f'Python Time: {time.time() - start:.5f} s')

_ = hist_numba(data, bins)
start = time.time()
_ = hist_numba(data, bins)
print(f'Numba Time: {time.time() - start:.5f} s')

In [ ]:
# Q3
import random
from numba import njit
import time

nsamples = 5_000_000

def monte_carlo_pi_py(nsamples):
    acc = 0
    for i in range(nsamples):
        x = random.random()
        y = random.random()
        if (x**2 + y**2) < 1.0:
            acc += 1
    return 4.0 * acc / nsamples

@njit
def monte_carlo_pi_nb(nsamples):
    acc = 0
    for i in range(nsamples):
        x = random.random()
        y = random.random()
        if (x**2 + y**2) < 1.0:
            acc += 1
    return 4.0 * acc / nsamples

start = time.time()
pi_py = monte_carlo_pi_py(nsamples)
t_py = time.time() - start
print(f'Python Time: {t_py:.5f} s, Pi={pi_py}')

start = time.time()
pi_nb = monte_carlo_pi_nb(nsamples)
t_nb_first = time.time() - start
print(f'Numba First Run Time: {t_nb_first:.5f} s')

start = time.time()
pi_nb2 = monte_carlo_pi_nb(nsamples)
t_nb = time.time() - start
print(f'Numba Second Run Time: {t_nb:.5f} s, Pi={pi_nb2}')
print(f'\nSpeedup (Python Time / Numba Second Time): {t_py/t_nb:.2f}x')

In [ ]:
# Q4
from numba import vectorize
import numpy as np
import time

N = 10_000_000
pixels = np.random.randint(0, 255, size=N, dtype=np.int64)

@vectorize(['int64(int64)'])
def adjust_brightness_seq(pixel):
    val = pixel * 1.2
    return 255 if val > 255 else int(val)

@vectorize(['int64(int64)'], target='parallel')
def adjust_brightness_par(pixel):
    val = pixel * 1.2
    return 255 if val > 255 else int(val)

_ = adjust_brightness_seq(pixels[:10])
_ = adjust_brightness_par(pixels[:10])

start = time.time()
res_seq = adjust_brightness_seq(pixels)
t_seq = time.time() - start
print(f'Sequential @vectorize Time: {t_seq:.5f} s')

start = time.time()
res_par = adjust_brightness_par(pixels)
t_par = time.time() - start
print(f'Parallel @vectorize Time: {t_par:.5f} s')

In [ ]:
# Q5
import numpy as np
from numba import njit
import time

N_samples = 100_000
N_features = 10
X = np.random.randn(N_samples, N_features)
y = np.random.choice([-1, 1], size=N_samples)
w = np.zeros(N_features)
lr = 0.01
epochs = 100

def logreg_numpy(X, y, w, lr, epochs):
    for _ in range(epochs):
        margin = y * np.dot(X, w)
        probs = 1 / (1 + np.exp(-margin))
        grad = -np.dot(X.T, y * (1 - probs)) / len(y)
        w -= lr * grad
    return w

@njit
def logreg_numba(X, y, w, lr, epochs):
    n = X.shape[0]
    for _ in range(epochs):
        grad = np.zeros_like(w)
        for i in range(n):
            margin = y[i] * np.dot(X[i], w)
            prob = 1.0 / (1.0 + np.exp(-margin))
            for j in range(len(w)):
                grad[j] -= X[i, j] * y[i] * (1.0 - prob)
        for j in range(len(w)):
            grad[j] /= n
            w[j] -= lr * grad[j]
    return w

w1 = np.zeros(N_features)
start = time.time()
w1_res = logreg_numpy(X, y, w1, lr, epochs)
print(f'NumPy Time: {time.time() - start:.5f} s')

w2 = np.zeros(N_features)
_ = logreg_numba(X[:10], y[:10], w2, lr, 1)
w2 = np.zeros(N_features)
start = time.time()
w2_res = logreg_numba(X, y, w2, lr, epochs)
print(f'Numba Time: {time.time() - start:.5f} s')

In [ ]:
# Q6
import numpy as np
from numba import cuda
import math

@cuda.jit
def matadd_kernel(A, B, C):
    row, col = cuda.grid(2)
    if row < A.shape[0] and col < A.shape[1]:
        C[row, col] = A[row, col] + B[row, col]

N = 1024
A = np.ones((N, N), dtype=np.float32)
B = np.ones((N, N), dtype=np.float32) * 2
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array((N, N), dtype=np.float32)
threadsperblock = (16, 16)
blockspergrid_x = math.ceil(A.shape[0] / threadsperblock[0])
blockspergrid_y = math.ceil(A.shape[1] / threadsperblock[1])
blockspergrid = (blockspergrid_x, blockspergrid_y)
matadd_kernel[blockspergrid, threadsperblock](d_A, d_B, d_C)
cuda.synchronize()
C = d_C.copy_to_host()
print('Top-left element of C:', C[0, 0])